In [1]:
# 配置依赖、数据路径、embedding 缓存、检索参数和随机种子。
import ast
import hashlib
import json
import random
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from openai import OpenAI
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, StandardScaler


DATA_FILE = Path("data/ROTD_final.xlsx")
RESULT_FILE = Path("results/use_case3_similar_case_retrieval.xlsx")
CACHE_DIR = Path("embedding_cache")
LLM_CONFIG_FILE = Path("LLM_config.json")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_FILE.parent.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL = "text-embedding-3-large"
EMBED_BATCH_SIZE = 32
MAX_RETRIES = 5
PCA_N_COMPONENTS = 32
RANDOM_STATE = 42
TOP_K_VALUES = [1, 5, 10]
EXAMPLE_QUERY_COUNT = 20

# 最终队列使用独立矩阵；旧缓存仅按相同模型和完整文本匹配复用。
RETRIEVAL_CACHE_TAG = "uc3_final_pastmed_reports_tumorsize"
EMBED_MATRIX_FILE = CACHE_DIR / f"embedding_matrix_{RETRIEVAL_CACHE_TAG}_{EMBEDDING_MODEL}.npy"
EMBED_META_FILE = CACHE_DIR / f"embedding_matrix_{RETRIEVAL_CACHE_TAG}_{EMBEDDING_MODEL}.meta.json"
EMBED_JSON_CACHE_FILE = CACHE_DIR / f"embedding_cache_{RETRIEVAL_CACHE_TAG}_{EMBEDDING_MODEL}.json"
FALLBACK_JSON_CACHE_FILES = [
    CACHE_DIR / f"embedding_cache_uc3_v6_pastmed_reports_tumorsize_{EMBEDDING_MODEL}.json",
    CACHE_DIR / f"embedding_cache_diagnosis_final_pastmed_reports_tumorsize_{EMBEDDING_MODEL}.json",
]

# 从 LLM_config.json 读取 API 配置，embedding 缺失时会用于调用接口。
def load_llm_config(config_file=LLM_CONFIG_FILE):
    if not Path(config_file).exists():
        raise FileNotFoundError(f"Missing LLM config file: {config_file}")
    with Path(config_file).open("r", encoding="utf-8-sig") as fh:
        config = json.load(fh)
    api_key = config.get("api_key") or config.get("OPENAI_API_KEY") or config.get("key")
    base_url = config.get("base_url") or config.get("OPENAI_BASE_URL") or config.get("url")
    if not api_key:
        raise ValueError(f"{config_file} 必须包含 api_key 或 OPENAI_API_KEY")
    return api_key, base_url

OPENAI_API_KEY, OPENAI_BASE_URL = load_llm_config()

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print(f"数据文件: {DATA_FILE}")
print(f"Embedding 矩阵文件: {EMBED_MATRIX_FILE}")
print(f"Embedding 模型: {EMBEDDING_MODEL}")
print(f"PCA 组件数: {PCA_N_COMPONENTS}")


数据文件: data\ROTD_final.xlsx
Embedding 矩阵文件: embedding_cache\embedding_matrix_uc3_final_pastmed_reports_tumorsize_text-embedding-3-large.npy
Embedding 模型: text-embedding-3-large
PCA 组件数: 32


In [2]:
# 定义文本拼接、embedding 缓存校验和按需生成 embedding 的工具函数。
# 将缺失值和常见占位文本统一为空字符串。
def normalize_text_value(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    text = str(value).strip()
    return "" if text in {"", "[]", "nan", "None", "NaN"} else text


# 将病史、报告和肿瘤大小拼接为检索用文本。
def build_retrieval_text(row):
    fields = {
        "past_med_hist": normalize_text_value(row.get("Past_medical_history")) or "N/A",
        "reports": normalize_text_value(row.get("Clinical_reports")) or "N/A",
        "tumor_size": normalize_text_value(row.get("Tumor_size")) or "N/A",
    }
    return "\n\n".join([f"{name}:\n{value}" for name, value in fields.items()])


# 对检索文本列表计算稳定哈希，用于校验 embedding 缓存是否匹配当前数据。
def hash_texts(texts):
    hasher = hashlib.sha256()
    for text in texts:
        encoded = str(text).encode("utf-8")
        hasher.update(len(encoded).to_bytes(8, "little"))
        hasher.update(encoded)
    return hasher.hexdigest()


# 读取 JSON 缓存文件，文件不存在时返回空字典。
def load_json(path):
    path = Path(path)
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


# 写出 JSON 缓存文件，并保留中文字符。
def save_json(path, payload):
    with Path(path).open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


# 读取 embedding 矩阵并用行数、模型名和文本哈希做一致性校验。
def load_embedding_matrix(matrix_file, meta_file, texts, model_name):
    matrix_file = Path(matrix_file)
    meta_file = Path(meta_file)
    if not matrix_file.exists():
        return None
    matrix = np.load(matrix_file).astype(np.float32)
    if matrix.shape[0] != len(texts):
        raise ValueError(f"Embedding 行数不匹配: {matrix.shape[0]} vs {len(texts)}")
    if meta_file.exists():
        meta = load_json(meta_file)
        if meta.get("model_name") != model_name:
            raise ValueError(f"{meta_file} 中的 embedding 模型不匹配")
        if meta.get("text_hash") != hash_texts(texts):
            print(f"Embedding text hash changed in {meta_file}; rebuilding from text cache.")
            return None
    else:
        print(f"警告：缺少 meta 文件 {meta_file}；仅使用行数校验。")
    print(f"已读取 embedding 矩阵: {matrix_file} {matrix.shape}")
    return matrix


# 保存 embedding 矩阵及其元数据，便于后续复跑时复用缓存。
def save_embedding_matrix(matrix_file, meta_file, matrix, texts, model_name):
    np.save(matrix_file, matrix.astype(np.float32))
    save_json(meta_file, {
        "model_name": model_name,
        "text_count": len(texts),
        "text_hash": hash_texts(texts),
        "shape": list(matrix.shape),
        "dtype": str(matrix.dtype),
        "data_file": str(DATA_FILE),
        "text_fields": ["Past_medical_history", "Clinical_reports", "Tumor_size"],
        "excluded_fields": ["Disease_name", "Disease_ids", "outcome labels", "Treatment_information", "stage", "Histopathological_subtype"],
    })


# 对未缓存文本批量调用 embedding API，并持续写入 JSON 缓存。
def generate_embeddings(texts, model_name, cache_file):
    cache = load_json(cache_file)
    missing_texts = set(texts) - cache.keys()
    for fallback_file in FALLBACK_JSON_CACHE_FILES:
        if not missing_texts:
            break
        fallback_cache = load_json(fallback_file)
        matched_texts = missing_texts & fallback_cache.keys()
        cache.update({text: fallback_cache[text] for text in matched_texts})
        missing_texts.difference_update(matched_texts)
        print(f"复用文本缓存: {fallback_file.name}, {len(matched_texts)} 条唯一文本")
        del fallback_cache
    vectors = [None] * len(texts)
    uncached = []
    uncached_idx = []
    for idx, text in enumerate(texts):
        if text in cache:
            vectors[idx] = np.asarray(cache[text], dtype=np.float32)
        else:
            uncached.append(text)
            uncached_idx.append(idx)

    print(f"文本数: {len(texts)}, 已缓存: {len(texts) - len(uncached)}, 待生成: {len(uncached)}")
    if uncached:
        if not OPENAI_API_KEY:
            raise RuntimeError(f"请在 {LLM_CONFIG_FILE} 中配置 api_key，用于生成缺失 embedding。")
        client_kwargs = {"api_key": OPENAI_API_KEY}
        if OPENAI_BASE_URL:
            client_kwargs["base_url"] = OPENAI_BASE_URL
        client = OpenAI(**client_kwargs)
    for start in range(0, len(uncached), EMBED_BATCH_SIZE):
        batch_texts = uncached[start:start + EMBED_BATCH_SIZE]
        batch_indices = uncached_idx[start:start + EMBED_BATCH_SIZE]
        last_error = None
        for attempt in range(MAX_RETRIES):
            try:
                response = client.embeddings.create(model=model_name, input=batch_texts)
                break
            except Exception as exc:
                last_error = exc
                time.sleep(min(2 ** attempt, 10))
        else:
            raise RuntimeError(f"Embedding 请求失败: {last_error}")

        if len(response.data) != len(batch_indices):
            raise ValueError("Embedding API 返回的向量数量与输入数量不一致")
        for row_idx, item in zip(batch_indices, sorted(response.data, key=lambda item: item.index)):
            vector = np.asarray(item.embedding, dtype=np.float32)
            vectors[row_idx] = vector
            cache[texts[row_idx]] = vector.tolist()
        save_json(cache_file, cache)
        print(f"已生成 {min(start + EMBED_BATCH_SIZE, len(uncached))}/{len(uncached)} 条未缓存文本的 embedding")
    matrix = np.vstack(vectors).astype(np.float32)
    if matrix.shape[1] != 3072 or not np.isfinite(matrix).all():
        raise ValueError("text-embedding-3-large 缓存维度错误或含非有限值")
    save_json(cache_file, cache)
    return matrix


In [3]:
# 定义最终队列的结构化特征工程和评价标签解析函数。
INVALID_CODE_VALUES = {
    "NA_NOT_DISEASE", "NA_OTHER_DISEASE",
    "NA_NOT_PHENOTYPE", "NA_NOT_LOCATION",
    "NA_EMPTY", "", "[]", "nan", "None", "NaN",
}


# 解析列表型编码字段，并去除无效占位编码。
def clean_code_list(value, invalid_values=INVALID_CODE_VALUES):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    text = str(value).strip()
    if text in invalid_values:
        return []
    try:
        parsed = ast.literal_eval(text)
        values = parsed if isinstance(parsed, (list, tuple, set)) else [parsed]
    except (ValueError, SyntaxError):
        values = text.replace(";", ",").split(",")
    cleaned = []
    seen = set()
    for item in values:
        code = str(item).strip()
        if code in invalid_values or code in seen:
            continue
        cleaned.append(code)
        seen.add(code)
    return cleaned


# 解析基因名称字段，支持列表字符串和逗号/分号分隔文本。
def parse_gene_list(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    text = str(value).strip()
    if text in {"", "[]", "nan", "None", "NaN"}:
        return []
    try:
        parsed = ast.literal_eval(text)
        values = parsed if isinstance(parsed, (list, tuple, set)) else [parsed]
    except (ValueError, SyntaxError):
        values = text.replace(";", ",").split(",")
    cleaned = sorted({str(item).strip() for item in values if str(item).strip() and str(item).strip() not in {"[]", "nan", "None", "NaN"}})
    return cleaned


# 创建兼容不同 scikit-learn 版本的 OneHotEncoder。
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


# 将年龄的年、月、日统一换算为年。
def calculate_age_years(row):
    y_val = pd.to_numeric(row.get("Age_y"), errors="coerce")
    m_val = pd.to_numeric(row.get("Age_m"), errors="coerce")
    d_val = pd.to_numeric(row.get("Age_d"), errors="coerce")
    if pd.isna(y_val) and pd.isna(m_val) and pd.isna(d_val):
        return np.nan
    return (0 if pd.isna(y_val) else y_val) + (0 if pd.isna(m_val) else m_val) / 12 + (0 if pd.isna(d_val) else d_val) / 365


# 将多标签字段转换为完整的 multi-hot 特征矩阵。
def transform_multilabel_full(series, parser, prefix):
    lists = series.apply(parser)
    classes = sorted({item for values in lists for item in values})
    if not classes:
        return np.zeros((len(series), 0), dtype=np.float32), [], lists
    mlb = MultiLabelBinarizer(classes=classes)
    matrix = mlb.fit_transform(lists).astype(np.float32)
    names = [f"{prefix}::{item}" for item in classes]
    return matrix, names, lists


# 构建年龄、人口学、表型、部位和遗传信息组成的结构化特征矩阵。
def build_structured_features(data):
    working = data.copy()
    genetic_cols = [
        "Genetic_testing_method", "Genotype", "Mutation_type",
        "Mutation_gene_name", "HGVS variant description", "Chromosome",
    ]
    for col in genetic_cols:
        if col in working.columns:
            working[col] = working[col].replace("[]", pd.NA)

    working["age"] = working.apply(calculate_age_years, axis=1)
    age_imputer = SimpleImputer(strategy="median")
    X_age = age_imputer.fit_transform(working[["age"]].values).astype(np.float32)

    cat_cols = ["Gender", "Ethnicity", "Country"]
    cat_df = working[cat_cols].fillna("Unknown").replace({"": "Unknown", "nan": "Unknown", "None": "Unknown"}).astype(str)
    ohe = make_one_hot_encoder()
    X_cat = ohe.fit_transform(cat_df).astype(np.float32)
    feature_names = ["age"] + list(ohe.get_feature_names_out(cat_cols))

    X_hpo, hpo_names, hpo_lists = transform_multilabel_full(working["Phenotype_ids"], clean_code_list, "hpo")
    X_location, location_names, location_lists = transform_multilabel_full(working["Anatomical_location_ids"], clean_code_list, "location")

    genetic_arrays = []
    genetic_feature_names = []
    genetic_parsed_lists = {}
    for col in genetic_cols:
        parser = parse_gene_list if col == "Mutation_gene_name" else clean_code_list
        X_genetic, names, lists = transform_multilabel_full(working[col], parser, col)
        genetic_arrays.append(X_genetic)
        genetic_feature_names.extend(names)
        genetic_parsed_lists[col] = lists

    feature_names.extend(hpo_names)
    feature_names.extend(location_names)
    feature_names.extend(genetic_feature_names)
    X_structured = np.hstack([X_age, X_cat, X_hpo, X_location, *genetic_arrays]).astype(np.float32)
    parsed_lists = {
        "hpo": hpo_lists,
        "location": location_lists,
        "gene": genetic_parsed_lists.get("Mutation_gene_name", pd.Series([[]] * len(working))),
        "genetics": genetic_parsed_lists,
    }
    return X_structured, feature_names, parsed_lists


# 解析 disease_codes，仅用于评价命中，不进入检索输入特征。
def parse_disease_codes(series):
    return series.apply(clean_code_list)


In [4]:
# 读取最终数据，先排除无可评价疾病编码的病例，再构造检索文本和独立 embedding 矩阵。
required_cols = [
    "pat_id", "PMID",
    "Age_y", "Age_m", "Age_d", "Gender", "Ethnicity", "Country",
    "Phenotype_ids", "Anatomical_location_ids",
    "Genetic_testing_method", "Genotype", "Mutation_type",
    "Mutation_gene_name", "HGVS variant description", "Chromosome",
    "Past_medical_history", "Clinical_reports", "Tumor_size",
    "Disease_name", "Disease_ids",
]

raw_data = pd.read_excel(DATA_FILE, dtype=str)
missing_cols = [col for col in required_cols if col not in raw_data.columns]
if missing_cols:
    raise ValueError(f"缺少必需列: {missing_cols}")
if raw_data["pat_id"].duplicated().any():
    duplicated = raw_data.loc[raw_data["pat_id"].duplicated(), "pat_id"].unique()[:10].tolist()
    raise ValueError(f"发现重复 pat_id: {duplicated}")

raw_disease_lists = parse_disease_codes(raw_data["Disease_ids"])
clean_disease_mask = raw_disease_lists.apply(len).gt(0)
excluded_no_disease_df = raw_data.loc[~clean_disease_mask, ["pat_id", "PMID", "Disease_name", "Disease_ids"]].copy()

data = raw_data.loc[clean_disease_mask].reset_index(drop=True).copy()
disease_lists = raw_disease_lists.loc[clean_disease_mask].reset_index(drop=True)
data["clean_disease_codes"] = disease_lists.apply(lambda values: json.dumps(values, ensure_ascii=False))

data["retrieval_text"] = data.apply(build_retrieval_text, axis=1)
texts_for_embedding = data["retrieval_text"].tolist()

embedding_matrix = load_embedding_matrix(EMBED_MATRIX_FILE, EMBED_META_FILE, texts_for_embedding, EMBEDDING_MODEL)
if embedding_matrix is None:
    print(f"未找到最终队列 embedding 矩阵: {EMBED_MATRIX_FILE}.")
    print(f"正在核对 {EMBEDDING_MODEL} 文本缓存并生成缺失 embedding...")
    embedding_matrix = generate_embeddings(texts_for_embedding, EMBEDDING_MODEL, EMBED_JSON_CACHE_FILE)
    save_embedding_matrix(EMBED_MATRIX_FILE, EMBED_META_FILE, embedding_matrix, texts_for_embedding, EMBEDDING_MODEL)
    print(f"已保存 embedding 矩阵: {EMBED_MATRIX_FILE} {embedding_matrix.shape}")

print(f"最终数据原始行数: {len(raw_data)}")
print(f"因无可评价疾病编码排除的行数: {len(excluded_no_disease_df)}")
print(f"进入检索的行数: {len(data)}")
print(f"Embedding 矩阵维度: {embedding_matrix.shape}")
display(data[["pat_id", "PMID", "Disease_name", "clean_disease_codes", "retrieval_text"]].head(3))


Embedding text hash changed in embedding_cache\embedding_matrix_uc3_final_pastmed_reports_tumorsize_text-embedding-3-large.meta.json; rebuilding from text cache.
未找到最终队列 embedding 矩阵: embedding_cache\embedding_matrix_uc3_final_pastmed_reports_tumorsize_text-embedding-3-large.npy.
正在核对 text-embedding-3-large 文本缓存并生成缺失 embedding...


复用文本缓存: embedding_cache_uc3_v6_pastmed_reports_tumorsize_text-embedding-3-large.json, 0 条唯一文本


复用文本缓存: embedding_cache_diagnosis_final_pastmed_reports_tumorsize_text-embedding-3-large.json, 0 条唯一文本
文本数: 2121, 已缓存: 1607, 待生成: 514


已生成 32/514 条未缓存文本的 embedding


已生成 64/514 条未缓存文本的 embedding


已生成 96/514 条未缓存文本的 embedding


... (12 outputs left out)

已生成 512/514 条未缓存文本的 embedding


已生成 514/514 条未缓存文本的 embedding


已保存 embedding 矩阵: embedding_cache\embedding_matrix_uc3_final_pastmed_reports_tumorsize_text-embedding-3-large.npy (2121, 3072)
最终数据原始行数: 2134
因无可评价疾病编码排除的行数: 13
进入检索的行数: 2121
Embedding 矩阵维度: (2121, 3072)


,pat_id,PMID,Disease_name,clean_disease_codes,retrieval_text
0,pat_26077847_01,26077847,"[""Bilateral diffuse uveal melanocytic prolifer...","[""ORPHA:674968"", ""ICD-11:2C25.0""]",past_med_hist:\nCataract surgery (no improveme...
1,pat_26077847_02,26077847,"[""Bilateral diffuse uveal melanocytic prolifer...","[""ORPHA:674968"", ""ICD-11:2C25.2""]",past_med_hist:\nNo past medical history (disea...
2,pat_26869128_01,26869128,"[""Ocular surface squamous neoplasia""]","[""ORPHA:659744""]",past_med_hist:\nNo past medical history (disea...


In [5]:
# 构建早期融合检索矩阵：结构化特征 + 标准化后 PCA(32) 文本 embedding，然后整体 StandardScaler。
X_structured, structured_feature_names, parsed_lists = build_structured_features(data)

embedding_scaler = StandardScaler()
X_embedding_scaled = embedding_scaler.fit_transform(embedding_matrix).astype(np.float32)
used_pca_components = min(PCA_N_COMPONENTS, X_embedding_scaled.shape[0] - 1, X_embedding_scaled.shape[1])
pca = PCA(n_components=used_pca_components, random_state=RANDOM_STATE)
X_embedding_pca = pca.fit_transform(X_embedding_scaled).astype(np.float32)

X_fusion = np.hstack([X_structured, X_embedding_pca]).astype(np.float32)
fusion_scaler = StandardScaler()
X_fusion_scaled = fusion_scaler.fit_transform(X_fusion).astype(np.float32)

print(f"结构化特征矩阵维度: {X_structured.shape}")
print(f"结构化特征数量: {len(structured_feature_names)}")
print(f"Embedding PCA 矩阵维度: {X_embedding_pca.shape}")
print(f"PCA 解释方差比例总和: {pca.explained_variance_ratio_.sum():.4f}")
print(f"标准化后融合矩阵维度: {X_fusion_scaled.shape}")


结构化特征矩阵维度: (2121, 2794)
结构化特征数量: 2794
Embedding PCA 矩阵维度: (2121, 32)
PCA 解释方差比例总和: 0.5882
标准化后融合矩阵维度: (2121, 2826)


In [6]:
# 计算 leave-one-PMID-out 余弦相似度，并生成每个 query 的 top-10 检索列表。
# 判断候选病例是否与查询病例共享任一疾病编码。
def shares_any_code(query_codes, candidate_codes):
    return bool(set(query_codes) & set(candidate_codes))


# 计算两个多标签集合的 Jaccard 重叠比例。
def jaccard_overlap(query_values, candidate_values):
    query_set = set(query_values)
    candidate_set = set(candidate_values)
    if not query_set and not candidate_set:
        return np.nan
    union = query_set | candidate_set
    if not union:
        return np.nan
    return len(query_set & candidate_set) / len(union)


# 计算 query 与 top-k 候选病例的平均多标签重叠比例。
def mean_topk_overlap(query_values, ranked_indices, candidate_lists, k):
    selected = ranked_indices[:k]
    values = [jaccard_overlap(query_values, candidate_lists[idx]) for idx in selected]
    values = [value for value in values if not pd.isna(value)]
    return np.nan if not values else float(np.mean(values))


disease_code_lists = disease_lists.tolist()
hpo_lists = parsed_lists["hpo"].tolist()
location_lists = parsed_lists["location"].tolist()
pmid_values = data["PMID"].astype(str).tolist()
pat_ids = data["pat_id"].astype(str).tolist()

similarity_matrix = cosine_similarity(X_fusion_scaled).astype(np.float32)
n_cases = len(data)
topk_by_query = []
support_rows = []

for query_idx in range(n_cases):
    valid_mask = np.ones(n_cases, dtype=bool)
    valid_mask[query_idx] = False
    valid_mask &= np.asarray(pmid_values) != pmid_values[query_idx]
    candidate_indices = np.flatnonzero(valid_mask)
    candidate_scores = similarity_matrix[query_idx, candidate_indices]
    order = np.argsort(-candidate_scores, kind="mergesort")
    ranked_indices = candidate_indices[order]
    ranked_scores = candidate_scores[order]
    topk_by_query.append((ranked_indices, ranked_scores))

    query_codes = disease_code_lists[query_idx]
    support_in_retrieval = sum(
        1 for candidate_idx in candidate_indices
        if shares_any_code(query_codes, disease_code_lists[candidate_idx])
    )
    support_rows.append({
        "query_index": query_idx,
        "pat_id": pat_ids[query_idx],
        "PMID": pmid_values[query_idx],
        "Disease_name": data.iloc[query_idx]["Disease_name"],
        "Disease_ids": json.dumps(query_codes, ensure_ascii=False),
        "retrieval_database_size": len(candidate_indices),
        "support_in_retrieval": support_in_retrieval,
    })

support_df = pd.DataFrame(support_rows)
print(support_df["support_in_retrieval"].ge(1).value_counts().rename({True: "排除同 PMID 后 support>=2", False: "排除同 PMID 后 support<2"}).to_string())
display(support_df.head())


support_in_retrieval
排除同 PMID 后 support>=2    1989
排除同 PMID 后 support<2      132


,query_index,pat_id,PMID,Disease_name,Disease_ids,retrieval_database_size,support_in_retrieval
0,0,pat_26077847_01,26077847,"[""Bilateral diffuse uveal melanocytic prolifer...","[""ORPHA:674968"", ""ICD-11:2C25.0""]",2119,39
1,1,pat_26077847_02,26077847,"[""Bilateral diffuse uveal melanocytic prolifer...","[""ORPHA:674968"", ""ICD-11:2C25.2""]",2119,39
2,2,pat_26869128_01,26869128,"[""Ocular surface squamous neoplasia""]","[""ORPHA:659744""]",2120,223
3,3,pat_27893585_01,27893585,"[""Conjunctival malignant melanoma""]","[""ORPHA:617910""]",2120,153
4,4,pat_29423784_01,29423784,"[""Ocular surface squamous neoplasia"", ""Fungal ...","[""ORPHA:659744"", ""ORPHA:519930""]",2120,223


In [7]:
# 生成每个 query 的核心指标，并汇总主检索结果。
query_metric_rows = []

for query_idx, (ranked_indices, ranked_scores) in enumerate(topk_by_query):
    query_codes = disease_code_lists[query_idx]
    support_in_retrieval = int(support_df.loc[query_idx, "support_in_retrieval"])
    analysis_group = "Support >= 2" if support_in_retrieval >= 1 else "Support < 2"

    relevant_positions = []
    if analysis_group == "Support >= 2":
        relevant_positions = [
            rank
            for rank, candidate_idx in enumerate(ranked_indices, start=1)
            if shares_any_code(query_codes, disease_code_lists[candidate_idx])
        ]
    first_relevant_rank = relevant_positions[0] if relevant_positions else np.nan
    top1_idx = int(ranked_indices[0]) if len(ranked_indices) else None
    top1_score = float(ranked_scores[0]) if len(ranked_scores) else np.nan

    row = {
        "query_index": query_idx,
        "pat_id": pat_ids[query_idx],
        "PMID": pmid_values[query_idx],
        "Disease_name": data.iloc[query_idx]["Disease_name"],
        "Disease_ids": json.dumps(query_codes, ensure_ascii=False),
        "analysis_group": analysis_group,
        "effective_support": support_in_retrieval,
        "retrieval_database_size": int(support_df.loc[query_idx, "retrieval_database_size"]),
        "hit@1": np.nan,
        "hit@5": np.nan,
        "hit@10": np.nan,
        "first_relevant_rank": first_relevant_rank,
        "mrr": np.nan if pd.isna(first_relevant_rank) else 1.0 / first_relevant_rank,
        "hpo_overlap@5": np.nan,
        "location_overlap@5": np.nan,
        "top1_pat_id": pat_ids[top1_idx] if top1_idx is not None else np.nan,
        "top1_disease_name": data.iloc[top1_idx]["Disease_name"] if top1_idx is not None else np.nan,
        "top1_similarity": top1_score,
    }

    # 所有 query 都有 top-5 候选，因此 HPO overlap@5 可在两个 support 分组中统一计算。
    row["hpo_overlap@5"] = mean_topk_overlap(hpo_lists[query_idx], ranked_indices, hpo_lists, 5)
    row["location_overlap@5"] = mean_topk_overlap(location_lists[query_idx], ranked_indices, location_lists, 5)

    if analysis_group == "Support >= 2":
        for k in TOP_K_VALUES:
            row[f"hit@{k}"] = int(any(
                shares_any_code(query_codes, disease_code_lists[candidate_idx])
                for candidate_idx in ranked_indices[:k]
            ))

    query_metric_rows.append(row)

query_metrics_df = pd.DataFrame(query_metric_rows)
evaluable_df = query_metrics_df[query_metrics_df["analysis_group"].eq("Support >= 2")].copy()
rare_df = query_metrics_df[query_metrics_df["analysis_group"].eq("Support < 2")].copy()

summary_df = pd.DataFrame([
    {
        "analysis_group": "Support >= 2",
        "method": "early_fusion_structured_plus_embedding_pca32",
        "n_retrieval_cases": len(data),
        "n_queries": len(evaluable_df),
        "Hit@1": evaluable_df["hit@1"].mean() if not evaluable_df.empty else np.nan,
        "Hit@5": evaluable_df["hit@5"].mean() if not evaluable_df.empty else np.nan,
        "Hit@10": evaluable_df["hit@10"].mean() if not evaluable_df.empty else np.nan,
        "MRR": evaluable_df["mrr"].mean() if not evaluable_df.empty else np.nan,
        "Mean first relevant rank": evaluable_df["first_relevant_rank"].mean() if not evaluable_df.empty else np.nan,
        "Median first relevant rank": evaluable_df["first_relevant_rank"].median() if not evaluable_df.empty else np.nan,
        "Mean HPO overlap@5": evaluable_df["hpo_overlap@5"].mean() if not evaluable_df.empty else np.nan,
        "Median HPO overlap@5": evaluable_df["hpo_overlap@5"].median() if not evaluable_df.empty else np.nan,
        "Mean location overlap@5": evaluable_df["location_overlap@5"].mean() if not evaluable_df.empty else np.nan,
        "Median location overlap@5": evaluable_df["location_overlap@5"].median() if not evaluable_df.empty else np.nan,
        "embedding_model": EMBEDDING_MODEL,
        "PCA_components": used_pca_components,
    },
    {
        "analysis_group": "Support < 2",
        "method": "early_fusion_structured_plus_embedding_pca32",
        "n_retrieval_cases": len(data),
        "n_queries": len(rare_df),
        "Hit@1": np.nan,
        "Hit@5": np.nan,
        "Hit@10": np.nan,
        "MRR": np.nan,
        "Mean first relevant rank": np.nan,
        "Median first relevant rank": np.nan,
        "Mean HPO overlap@5": rare_df["hpo_overlap@5"].mean() if not rare_df.empty else np.nan,
        "Median HPO overlap@5": rare_df["hpo_overlap@5"].median() if not rare_df.empty else np.nan,
        "Mean location overlap@5": rare_df["location_overlap@5"].mean() if not rare_df.empty else np.nan,
        "Median location overlap@5": rare_df["location_overlap@5"].median() if not rare_df.empty else np.nan,
        "embedding_model": EMBEDDING_MODEL,
        "PCA_components": used_pca_components,
    },
])

display(summary_df)
display(query_metrics_df.head())


,analysis_group,method,n_retrieval_cases,n_queries,Hit@1,Hit@5,Hit@10,MRR,Mean first relevant rank,Median first relevant rank,Mean HPO overlap@5,Median HPO overlap@5,Mean location overlap@5,Median location overlap@5,embedding_model,PCA_components
0,Support >= 2,early_fusion_structured_plus_embedding_pca32,2121,1989,0.419306,0.728507,0.824032,0.55462,30.95274,2.0,0.141556,0.114087,0.256683,0.200000,text-embedding-3-large,32
1,Support < 2,early_fusion_structured_plus_embedding_pca32,2121,132,NaN,NaN,NaN,NaN,NaN,NaN,0.127863,0.111369,0.283374,0.166667,text-embedding-3-large,32


,query_index,pat_id,PMID,Disease_name,Disease_ids,analysis_group,effective_support,retrieval_database_size,hit@1,hit@5,hit@10,first_relevant_rank,mrr,hpo_overlap@5,location_overlap@5,top1_pat_id,top1_disease_name,top1_similarity
0,0,pat_26077847_01,26077847,"[""Bilateral diffuse uveal melanocytic prolifer...","[""ORPHA:674968"", ""ICD-11:2C25.0""]",Support >= 2,39,2119,1.0,1.0,1.0,1.0,1.000000,0.206061,0.208571,pat_32703386_01,"[""Bilateral diffuse uveal melanocytic prolifer...",0.369187
1,1,pat_26077847_02,26077847,"[""Bilateral diffuse uveal melanocytic prolifer...","[""ORPHA:674968"", ""ICD-11:2C25.2""]",Support >= 2,39,2119,0.0,1.0,1.0,4.0,0.250000,0.149487,0.066667,pat_31097438_02,"[""Uveal melanoma"", ""Impairment of binocular fu...",0.304816
2,2,pat_26869128_01,26869128,"[""Ocular surface squamous neoplasia""]","[""ORPHA:659744""]",Support >= 2,223,2120,0.0,1.0,1.0,3.0,0.333333,0.174286,0.250000,pat_26378741_05,"[""Retinoblastoma""]",0.474529
3,3,pat_27893585_01,27893585,"[""Conjunctival malignant melanoma""]","[""ORPHA:617910""]",Support >= 2,153,2120,0.0,1.0,1.0,3.0,0.333333,0.083333,0.170000,pat_36617852_02,"[""Adenoid cystic carcinoma""]",0.395272
4,4,pat_29423784_01,29423784,"[""Ocular surface squamous neoplasia"", ""Fungal ...","[""ORPHA:659744"", ""ORPHA:519930""]",Support >= 2,223,2120,1.0,1.0,1.0,1.0,1.000000,0.087879,0.233333,pat_38649476_01,"[""Ocular surface squamous neoplasia""]",0.444226


In [8]:
# 生成精简 cohort flow，用于说明最终队列如何经预处理进入主评价和稀有病例分析。
clean_code_counts = disease_lists.apply(len)
pmid_group_sizes = data.groupby(data["PMID"].astype(str)).size()

cohort_flow_df = pd.DataFrame([
    {
        "category": "最终数据原始病例总数",
        "n": len(raw_data),
        "definition": "data/ROTD_final.xlsx 在用例 3 预处理前的全部行。",
    },
    {
        "category": "排除无可评价疾病编码病例",
        "n": len(excluded_no_disease_df),
        "definition": "去除 NA_NOT_DISEASE 和 NA_OTHER_DISEASE 后 disease_codes 为空，因此不进入 embedding 和检索。",
    },
    {
        "category": "进入检索的病例",
        "n": len(data),
        "definition": "至少保留一个清洗后疾病编码的病例；embedding 和检索都在该队列上运行。",
    },
    {
        "category": "Support >= 2 查询",
        "n": len(evaluable_df),
        "definition": "排除自身和同 PMID 病例后，仍至少有一个相同疾病编码候选病例的查询。",
    },
    {
        "category": "Support < 2 查询",
        "n": len(rare_df),
        "definition": "leave-one-PMID-out 后没有相同疾病编码候选病例的查询；用 HPO/location 重叠和代表性案例辅助评估。",
    },
    {
        "category": "保留的唯一 PMID 数",
        "n": int(data["PMID"].astype(str).nunique()),
        "definition": "进入检索队列后保留的唯一 PubMed ID 数量。",
    },
    {
        "category": "包含多个病例的 PMID 组",
        "n": int((pmid_group_sizes > 1).sum()),
        "definition": "包含多个病人级记录的 PMID 组；每个查询检索时会排除同 PMID 病例。",
    },
])

display(cohort_flow_df)


,category,n,definition
0,最终数据原始病例总数,2134,data/ROTD_final.xlsx 在用例 3 预处理前的全部行。
1,排除无可评价疾病编码病例,13,去除 NA_NOT_DISEASE 和 NA_OTHER_DISEASE 后 disease...
2,进入检索的病例,2121,至少保留一个清洗后疾病编码的病例；embedding 和检索都在该队列上运行。
3,Support >= 2 查询,1989,排除自身和同 PMID 病例后，仍至少有一个相同疾病编码候选病例的查询。
4,Support < 2 查询,132,leave-one-PMID-out 后没有相同疾病编码候选病例的查询；用 HPO/loca...
5,保留的唯一 PMID 数,1019,进入检索队列后保留的唯一 PubMed ID 数量。
6,包含多个病例的 PMID 组,269,包含多个病人级记录的 PMID 组；每个查询检索时会排除同 PMID 病例。


In [9]:
# 生成代表性 top-5 检索案例，控制为最多 20 个 query，便于人工检查和论文补充材料。
example_rows = []
rare_query_indices = rare_df["query_index"].head(10).tolist()
failed_query_indices = evaluable_df[evaluable_df["hit@5"].eq(0)].sort_values("mrr", na_position="last")["query_index"].head(5).tolist()
success_query_indices = evaluable_df[evaluable_df["hit@5"].eq(1)].sort_values("mrr", ascending=False)["query_index"].head(5).tolist()
example_query_indices = rare_query_indices + failed_query_indices + success_query_indices

for query_idx in example_query_indices:
    ranked_indices, ranked_scores = topk_by_query[query_idx]
    analysis_group = query_metrics_df.loc[query_idx, "analysis_group"]
    for rank, (candidate_idx, score) in enumerate(zip(ranked_indices[:5], ranked_scores[:5]), start=1):
        example_rows.append({
            "query_pat_id": pat_ids[query_idx],
            "query_disease_name": data.iloc[query_idx]["Disease_name"],
            "query_disease_codes": json.dumps(disease_code_lists[query_idx], ensure_ascii=False),
            "analysis_group": analysis_group,
            "rank": rank,
            "candidate_pat_id": pat_ids[candidate_idx],
            "candidate_disease_name": data.iloc[candidate_idx]["Disease_name"],
            "candidate_disease_codes": json.dumps(disease_code_lists[candidate_idx], ensure_ascii=False),
            "cosine_similarity": float(score),
            "shares_disease_code": shares_any_code(disease_code_lists[query_idx], disease_code_lists[candidate_idx]),
            "hpo_jaccard": jaccard_overlap(hpo_lists[query_idx], hpo_lists[candidate_idx]),
            "location_jaccard": jaccard_overlap(location_lists[query_idx], location_lists[candidate_idx]),
        })

representative_top5_df = pd.DataFrame(example_rows)
display(representative_top5_df.head(15))


,query_pat_id,query_disease_name,query_disease_codes,analysis_group,rank,candidate_pat_id,candidate_disease_name,candidate_disease_codes,cosine_similarity,shares_disease_code,hpo_jaccard,location_jaccard
0,pat_26843453_02,"[""Carcinoma in situ, NOS""]","[""ICD-11:XH5NV6""]",Support < 2,1,pat_38456662_01,"[""Ocular surface squamous neoplasia""]","[""ORPHA:659744""]",0.127683,False,0.111111,0.500000
1,pat_26843453_02,"[""Carcinoma in situ, NOS""]","[""ICD-11:XH5NV6""]",Support < 2,2,pat_31193777_02,"[""Conjunctival malignant melanoma""]","[""ORPHA:617910""]",0.098381,False,0.000000,1.000000
2,pat_26843453_02,"[""Carcinoma in situ, NOS""]","[""ICD-11:XH5NV6""]",Support < 2,3,pat_31193777_01,"[""Conjunctival malignant melanoma""]","[""ORPHA:617910""]",0.092354,False,0.000000,1.000000
3,pat_26843453_02,"[""Carcinoma in situ, NOS""]","[""ICD-11:XH5NV6""]",Support < 2,4,pat_38990387_04,"[""Conjunctival malignant melanoma""]","[""ORPHA:617910""]",0.086741,False,0.000000,0.000000
4,pat_26843453_02,"[""Carcinoma in situ, NOS""]","[""ICD-11:XH5NV6""]",Support < 2,5,pat_30312574_04,"[""Conjunctival malignant melanoma""]","[""ORPHA:617910""]",0.085794,False,0.000000,1.000000
5,pat_26843453_03,"[""Carcinoma in situ, NOS""]","[""ICD-11:XH5NV6""]",Support < 2,1,pat_35959160_02,"[""Conjunctival malignant melanoma""]","[""ORPHA:617910""]",0.577459,False,0.250000,0.000000
6,pat_26843453_03,"[""Carcinoma in situ, NOS""]","[""ICD-11:XH5NV6""]",Support < 2,2,pat_35959160_01,"[""Conjunctival malignant melanoma""]","[""ORPHA:617910""]",0.500760,False,0.200000,0.000000
7,pat_26843453_03,"[""Carcinoma in situ, NOS""]","[""ICD-11:XH5NV6""]",Support < 2,3,pat_28700252_01,"[""Limbal stem cell deficiency""]","[""ORPHA:171673""]",0.320169,False,0.142857,0.000000
8,pat_26843453_03,"[""Carcinoma in situ, NOS""]","[""ICD-11:XH5NV6""]",Support < 2,4,pat_28275607_02,"[""Ocular surface squamous neoplasia""]","[""ORPHA:659744""]",0.238430,False,0.111111,0.333333
9,pat_26843453_03,"[""Carcinoma in situ, NOS""]","[""ICD-11:XH5NV6""]",Support < 2,5,pat_30633894_06,"[""Ocular surface squamous neoplasia""]","[""ORPHA:659744""]",0.215844,False,0.000000,1.000000


In [10]:
# 写出精简 workbook：Summary、Cohort_flow、Query_metrics、Representative_top5 和 Run_config。
# 写入 Excel 前统一整理为论文表格格式：Recall 命名改为 Hit，并把比例指标转为 xx.xx%。
UC3_RENAME = {"hit@1": "Hit@1", "hit@5": "Hit@5", "hit@10": "Hit@10", "Recall@1": "Hit@1", "Recall@5": "Hit@5", "Recall@10": "Hit@10"}
UC3_PERCENT_COLS = [
    "Hit@1",
    "Hit@5",
    "Hit@10",
    "MRR",
    "mrr",
    "Mean HPO overlap@5",
    "Median HPO overlap@5",
    "Mean location overlap@5",
    "Median location overlap@5",
    "hpo_overlap@5",
    "location_overlap@5",
    "hpo_jaccard",
    "location_jaccard",
]

def format_percent_value(value):
    if value is None or pd.isna(value):
        return ""
    if isinstance(value, str):
        text = value.strip()
        if not text or text.lower() in {"nan", "none", "na", "n/a"}:
            return ""
        if text.endswith("%"):
            return text
        value = float(text)
    return f"{float(value):.2%}"

def make_retrieval_paper_ready(df):
    df = df.copy().rename(columns=UC3_RENAME)
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].map(lambda value: value.replace("Recall@", "Hit@") if isinstance(value, str) else value)
    for col in UC3_PERCENT_COLS:
        if col in df.columns:
            df[col] = df[col].map(format_percent_value)
    return df

run_config_df = pd.DataFrame([
    {"parameter": "data_file", "value": str(DATA_FILE)},
    {"parameter": "raw_final_rows", "value": len(raw_data)},
    {"parameter": "excluded_without_evaluable_disease_code", "value": len(excluded_no_disease_df)},
    {"parameter": "retrieval_rows", "value": len(data)},
    {"parameter": "embedding_model", "value": EMBEDDING_MODEL},
    {"parameter": "embedding_matrix_file", "value": str(EMBED_MATRIX_FILE)},
    {"parameter": "embedding_text_fields", "value": "Past_medical_history, Clinical_reports, Tumor_size"},
    {"parameter": "excluded_input_fields", "value": "Disease_name, Disease_ids, 结局标签, Treatment_information, stage, Histopathological_subtype"},
    {"parameter": "structured_features", "value": "age, Gender, Ethnicity, Country, Phenotype_ids, Anatomical_location_ids, Genetic_testing_method, Genotype, Mutation_type, Mutation_gene_name, HGVS variant description, Chromosome"},
    {"parameter": "pca_requested_components", "value": PCA_N_COMPONENTS},
    {"parameter": "pca_used_components", "value": used_pca_components},
    {"parameter": "pca_explained_variance_ratio_sum", "value": float(pca.explained_variance_ratio_.sum())},
    {"parameter": "fusion_method", "value": "早期融合：结构化特征 + PCA embedding 特征"},
    {"parameter": "standardization", "value": "对完整无监督融合矩阵执行 StandardScaler fit_transform"},
    {"parameter": "similarity_method", "value": "余弦相似度"},
    {"parameter": "exclusion_rule", "value": "排除查询病例自身和所有同 PMID 病例"},
    {"parameter": "support_definition", "value": "Support >= 2 表示排除同 PMID 后仍至少有一个相同疾病编码候选病例；Support < 2 表示没有此类候选病例"},
    {"parameter": "example_selection", "value": "最多选择 10 个 Support < 2 查询、5 个 Hit@5 失败的 Support >= 2 查询，以及 5 个 Hit@5 成功的 Support >= 2 查询"},
    {"parameter": "random_state", "value": RANDOM_STATE},
])

summary_paper_df = make_retrieval_paper_ready(summary_df)
query_metrics_paper_df = make_retrieval_paper_ready(query_metrics_df.drop(columns=["query_index"]))
representative_top5_paper_df = make_retrieval_paper_ready(representative_top5_df)

with pd.ExcelWriter(RESULT_FILE, engine="openpyxl") as writer:
    summary_paper_df.to_excel(writer, sheet_name="All_results", index=False)
    cohort_flow_df.to_excel(writer, sheet_name="Cohort_flow", index=False)
    query_metrics_paper_df.to_excel(writer, sheet_name="Query_metrics", index=False)
    representative_top5_paper_df.to_excel(writer, sheet_name="Representative_top5", index=False)
    run_config_df.to_excel(writer, sheet_name="Run_config", index=False)

print(f"Saved results to {RESULT_FILE}")


Saved results to results\use_case3_similar_case_retrieval.xlsx
